# 05 — Correlation Networks & Eigenvalue Analysis

We:
1. Build a gene–gene correlation matrix
2. Perform eigenvalue decomposition to find latent structure
3. Apply the Marchenko–Pastur law to distinguish signal from noise
4. Construct and visualise a gene co-expression network
5. Identify network hubs (potential biomarkers)

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = '/content/gene-expression-la-analysis'
sys.path.insert(0, f'{PROJECT_ROOT}/src')

from linear_algebra import (
    gene_correlation_matrix, sample_correlation_matrix,
    eigenvalue_decomposition, marchenko_pastur_threshold,
    significant_eigengenes
)
from visualization import (
    plot_correlation_heatmap, plot_eigenvalue_spectrum,
    plot_correlation_network, plot_gene_loadings
)

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Ready.')


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────
expr = pd.read_csv(f'{PROJECT_ROOT}/data/processed/processed_expression.csv',
                   index_col=0)
meta = pd.read_csv(f'{PROJECT_ROOT}/data/processed/metadata_clean.csv',
                   index_col=0)

# Use top 1000 most-variable genes for network (computational speed)
top_genes = expr.var(axis=1).nlargest(1000).index
expr_top  = expr.loc[top_genes]
print(f'Using top {len(top_genes)} variable genes')


## 1. Gene–Gene Correlation Matrix

In [ ]:
corr_gene = gene_correlation_matrix(expr_top, method='pearson')
print('Correlation matrix:', corr_gene.shape)


In [ ]:
# Distribution of correlation values
upper = corr_gene.where(np.triu(np.ones(corr_gene.shape), k=1).astype(bool))
upper_vals = upper.stack().values

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(upper_vals, bins=100, color='steelblue', edgecolor='white')
ax.axvline(0.7,  ls='--', color='crimson',  label='r=0.7')
ax.axvline(-0.7, ls='--', color='darkblue', label='r=-0.7')
ax.set_xlabel('Pearson r')
ax.set_ylabel('Count')
ax.set_title('Distribution of Gene–Gene Correlations')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/05_corr_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Heatmap of top-200 genes (for visibility)
top200 = expr_top.var(axis=1).nlargest(200).index
plot_correlation_heatmap(corr_gene.loc[top200, top200],
                          title='Gene–Gene Pearson Correlation (top 200 variable genes)',
                          save_path=f'{PROJECT_ROOT}/data/results/05_corr_heatmap.png')


## 2. Eigenvalue Decomposition & Marchenko–Pastur

In [ ]:
eig_result = eigenvalue_decomposition(corr_gene)

n_genes, n_samples = expr_top.shape
mp_thresh = marchenko_pastur_threshold(n_genes, n_samples, sigma=1.0)

sig_idx = significant_eigengenes(eig_result, threshold=mp_thresh)
print(f'Significant eigengenes (above M-P threshold): {len(sig_idx)}')


In [ ]:
# Show top 50 eigenvalues
plot_eigenvalue_spectrum(eig_result['eigenvalues'][:50],
                          mp_threshold=mp_thresh,
                          title='Eigenvalue Spectrum (top 50)',
                          save_path=f'{PROJECT_ROOT}/data/results/05_eigenvalue_spectrum.png')


In [ ]:
# ── Cumulative variance of eigenvalues ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.cumsum(eig_result['explained_variance_ratio']) * 100, color='steelblue')
ax.axhline(90, ls='--', color='grey', label='90%')
ax.set_xlabel('Eigengene rank')
ax.set_ylabel('Cumulative Variance (%)')
ax.set_title('Cumulative Variance — Gene Correlation Eigengenes')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/05_cum_variance.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 3. Gene Co-Expression Network

In [ ]:
# Use top 100 most-variable genes for the network (readable visualisation)
top100 = expr_top.var(axis=1).nlargest(100).index
corr_net = corr_gene.loc[top100, top100]

plot_correlation_network(corr_net, threshold=0.70,
                          title='Gene Co-Expression Network',
                          save_path=f'{PROJECT_ROOT}/data/results/05_network.png')


In [ ]:
# ── Hub genes (highest degree) ────────────────────────────────────────────
import networkx as nx

G = nx.Graph()
genes = corr_net.index.tolist()
G.add_nodes_from(genes)
THRESHOLD = 0.70

for i in range(len(genes)):
    for j in range(i+1, len(genes)):
        if abs(corr_net.iloc[i, j]) >= THRESHOLD:
            G.add_edge(genes[i], genes[j], weight=corr_net.iloc[i, j])

degree = dict(G.degree())
hub_df = pd.Series(degree, name='degree').sort_values(ascending=False)
print('Top 20 hub genes:')
print(hub_df.head(20).to_string())
hub_df.to_csv(f'{PROJECT_ROOT}/data/results/05_hub_genes.csv')


In [ ]:
# ── Top eigengene weights ──────────────────────────────────────────────────
top_eigengene = pd.Series(eig_result['eigenvectors'][:, 0],
                           index=eig_result['labels'],
                           name='Eigengene1')

plot_gene_loadings(top_eigengene, top_n=30,
                   title='Top Gene Weights — 1st Eigengene',
                   save_path=f'{PROJECT_ROOT}/data/results/05_eigengene1_weights.png')


In [ ]:
# ── Save eigenvectors ─────────────────────────────────────────────────────
n_save = min(20, len(sig_idx))
eig_df = pd.DataFrame(
    eig_result['eigenvectors'][:, :n_save],
    index=eig_result['labels'],
    columns=[f'Eigengene{i+1}' for i in range(n_save)]
)
eig_df.to_csv(f'{PROJECT_ROOT}/data/processed/eigengenes.csv')
print(f'Saved top-{n_save} eigengenes.')
